# UCB-FE Experiments

This notebooks provides some neccessary calculations for the UCB-FE experiments. Logic about using ML models are implemeted in the [utils file](./utils_ucb_fe.py).

The following code loads the data, train models, if the models are not already trained, and then evaluate the models. Data analysis is done in the next notebooks.

In [ ]:
import numpy  as np
import pandas as pd
import warnings


from pandas.errors import SettingWithCopyWarning
from utils_ucb_fe import ML, TabMModel, Lightgbm, Xgboost, Catboost, Tab_net
from sklearn.metrics import roc_auc_score
from tqdm import tqdm


warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=RuntimeWarning) 
warnings.filterwarnings("ignore", category=SettingWithCopyWarning)

%load_ext autoreload
%autoreload 2

In [ ]:
# Read data

# Read data
train_df = pd.read_parquet('data/train_int.parquet')
test_df = pd.read_parquet('data/test_int.parquet')[:]
train_df = train_df.loc[(train_df['item_imps'] > 0)]
# df_base = test_df[:]
# df = test_df[:]

y_train = train_df.target
X_train = train_df.drop(['target', 'request_id', 'item_id', 'item_imps', 'item_shows'], axis=1)

y_test = test_df.target
X_test = test_df.drop(['target', 'request_id', 'item_id', 'item_imps', 'item_shows'], axis=1)

categorical_features = [ "mcat", "mcat_1", "mcat_2", "mcat_3", "mcat_4", "mcat_5", "cat_id", "item_region_id", "item_location_id" ]
df = test_df[['target', 'request_id', 'item_id', 'item_imps', 'ctr']]


model_dict = {
     'lightgbm': lightgbm_ml,
    'xgboost': xgboost_ml,
    'tabm': tabm_ml, 
    'catboost': catboost_ml, 
    'tabnet': tabnet_ml
}

model_names = model_dict.keys()

In [ ]:
# Init models

tabm_ml = TabMModel(X_train, y_train, categorical_features)
lightgbm_ml = Lightgbm(X_train, y_train, categorical_features)
xgboost_ml = Xgboost(X_train, y_train, categorical_features)
catboost_ml = Catboost(X_train, y_train, categorical_features)
tabnet_ml = Tab_net(X_train, y_train, categorical_features)

model_dict = {
     'lightgbm': lightgbm_ml,
    'xgboost': xgboost_ml,
    'tabm': tabm_ml, 
    'catboost': catboost_ml, 
    'tabnet': tabnet_ml
}
model_names = model_dict.keys()

for model_name in model_names:
    ml = model_dict[model_name]
    try:
        ml._load_model_()
        print(f"Loaded pre-trained {model_name} model")
    except FileNotFoundError:
        print(f"No pre-trained model found for {model_name}. Fitting the model.")
        ml.fit()

In [ ]:
#params
cold_periods = [0, 10, 100, 200, 500]
delta = 1.5
top_n = 10
tail_m = 30

In [ ]:
def ucb_ctr_task(df, delta, T):

    """
    ----------------------------------------------------------------------------------------
    delta : param of UCB 
    T : cold-start period   
    ----------------------------------------------------------------------------------------

    Create additional columns displaying the value of the !CTR feature! according to the UCB approach
 
    e.g., column 'ucb_ctr_T_100' shows UCB value of !CTR feature! that incorporates a cold-start period of T=100.

    e.g., column 'cold_100' indicates (1/0) if an item falls within the cold-start period of T=100.
    """
    
    N = np.clip(df["item_imps"], a_min = 0.5, a_max = None)
    
    df[f"cold_{T}"] = 0 + (df['item_imps'] <= T)
    
    if T <= 1:
        df[f'ucb_ctr_T_{T}'] = df[f'cold_{T}'] + (1- df[f'cold_{T}'])* df['ctr'] 
    else:
        df['ucb_ctr_T_'+str(T)] = np.clip(df['cold_'+str(T)] * (df['ctr'] + np.sqrt(delta * np.log(T) / N)) + (1- df['cold_'+str(T)])* df['ctr'], a_min = 0, a_max = 1)


def ctr_prediction_task(df: pd.DataFrame, X_test: pd.DataFrame, T:float, ml:ML, model_name:str):

    """
    ----------------------------------------------------------------------------------------
    T : cold-start period   
    model_name : name of the ML model, selected from a predefined list of pre-trained models
    ----------------------------------------------------------------------------------------

    Create additional columns displaying the value of the !CTR prediction!
 
    e.g., column 'base_ctr_pred_catboost' shows CatBoost baseline model's CTR predictions

    e.g., column 'ucb_ctr_pred_T_100_catboost' shows CatBoost model's CTR predictions that incorporates UCB-FE and a cold-start period of T=100
    """

    #baseline ctr prediction
    X_test['ctr'] = df['ctr'].values
    df[f"base_ctr_pred_{model_name}"] = ml.predict(X_test) #ml.predict(X_test) ml.ctr
    
    #fe-ucb ctr prediction
    X_test['ctr'] = df[f"ucb_ctr_T_{T}"].values
    df[f"ucb_ctr_pred_T_{T}_{model_name}"] = ml.predict(X_test) # ml.predict(X_test) ml.ctr

for T in cold_periods:
    ucb_ctr_task(df, delta, T)
    for model_name in model_names:
        ml = model_dict[model_name]
        ctr_prediction_task(df, X_test, T, ml, model_name)


In [ ]:
def add_position(df, model_names, cold_periods):

    """
    ----------------------------------------------------------------------------------------
    model_names : predefined list of names of pre-trained models [catboost, xgboost,...]
    cold_periods : list of values of cold-start periods [0, 10, 100,...]
    ----------------------------------------------------------------------------------------

    Create additional columns displaying SERP positions ranked by predicted CTR.

    e.g., column 'pos_old_catboost' shows rankings based on the CatBoost baseline model's predictions.

    e.g., column 'pos_new_T_100_catboost' shows rankings from a CatBoost model that incorporates UCB-FE and a cold-start period of T=100.
    """

    codes, _ = pd.factorize(df['request_id'])
    df['request_id'] = codes
    df = df.sort_values(['request_id'], ascending=[True])
    
    n_requests = df['request_id'].nunique()
    pos_serp = []
    n_items = []


    for serp_x in range(n_requests):
        df_serp_x = df[df['request_id'] == serp_x]
        n_items_serp_x = df_serp_x.shape[0]
        n_items += [n_items_serp_x] * n_items_serp_x
        pos_serp += range(n_items_serp_x)

    for name in model_names:
        df = df.sort_values(['request_id', 'base_ctr_pred_' + name], ascending=[True, False])
        df['pos_old_'+ name] = pos_serp
        df['n_items'] = n_items

        for T in cold_periods:
            df = df.sort_values(['request_id', 'ucb_ctr_pred_T_'+str(T) + '_' + name], ascending=[True, False])
            df['pos_new_T_'+str(T) + '_' + name] = pos_serp
    
    return df

df = add_position(df, model_names, cold_periods)

df.to_parquet('data/results/df_ucb_fe.parquet')